# 07 — SOFA horario: artefacto, completitud y distribución

Este notebook materializa (si es necesario) y valida el artefacto `30_score/sofa_hourly` de una ejecución versionada bajo `data/derived/sofa/<run_id>/`. Solo presenta **resultados agregados**: nunca imprime identificadores ni filas a nivel de paciente. Todavía no asigna Sepsis-3 ni shock; ese enlace requiere terminar y validar el tiempo de sospecha de infección.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile

import pandas as pd
from IPython.display import SVG, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from mimic_sepsis.artifacts import ArtifactStore

ARTIFACT_NAME = 'sofa_hourly'
SOFA_RUNS_ROOT = PROJECT_ROOT / 'data' / 'derived' / 'sofa'
BUILDER = PROJECT_ROOT / 'scripts' / 'build_demo_sofa_incremental.py'
DATA_VERSION = '2.2'

def available_score_stores():
    """Devuelve ejecuciones completas ordenadas por fecha y ruta."""
    candidates = []
    for manifest_path in SOFA_RUNS_ROOT.glob('*/30_score/sofa_hourly.manifest.json'):
        store = ArtifactStore(manifest_path.parent)
        try:
            manifest = store.validate(ARTIFACT_NAME)
        except Exception as exc:
            print(f'Se omite artefacto inválido {manifest_path}: {exc}')
            continue
        candidates.append((manifest.created_at_utc, str(manifest_path.parent), store))
    return sorted(candidates, key=lambda item: (item[0], item[1]))


## Materialización incremental

La lógica clínica vive en `src/` y el script versionado. Si el artefacto final no existe, la siguiente celda ejecuta el constructor desde la raíz del repositorio. Un fallo conserva el error completo y ofrece el comando exacto para reproducirlo en terminal.

In [ ]:
candidates = available_score_stores()
if not candidates:
    if not BUILDER.exists():
        raise FileNotFoundError(
            f'Falta {BUILDER}. Restaura el script versionado antes de continuar.'
        )
    command = [sys.executable, str(BUILDER)]
    completed = subprocess.run(
        command, cwd=PROJECT_ROOT, text=True, capture_output=True
    )
    if completed.returncode != 0:
        raise RuntimeError(
            'No se pudo construir el SOFA incremental. Ejecuta desde la raíz:\n'
            f'  {sys.executable} scripts/build_demo_sofa_incremental.py\n\n'
            f'stdout:\n{completed.stdout}\n\nstderr:\n{completed.stderr}'
        )
    print(completed.stdout.strip() or 'Artefacto construido.')
    candidates = available_score_stores()
    if not candidates:
        raise RuntimeError('El constructor terminó pero no publicó 30_score/sofa_hourly.')

# Selección determinista: fecha UTC del manifiesto y ruta como desempate.
# Los run_id son hashes de configuración; normalmente existirá un único candidato.
_, _, STORE = candidates[-1]
print(f'Usando {STORE.root.relative_to(PROJECT_ROOT)} ({len(candidates)} ejecución/es válida/s)')


## Integridad y procedencia

Se comprueban checksum, esquema y número de filas antes de leer. La salida se limita a metadatos no identificables.

In [ ]:
manifest = STORE.validate(ARTIFACT_NAME)
assert manifest.data_version == DATA_VERSION, (
    f'Versión inesperada: {manifest.data_version!r}; se esperaba {DATA_VERSION!r}'
)
pd.DataFrame([{
    'artifact': manifest.artifact,
    'data_version': manifest.data_version,
    'code_version': manifest.code_version,
    'rows': manifest.rows,
    'n_columns': len(manifest.columns),
    'created_at_utc': manifest.created_at_utc,
    'sha256_prefix': manifest.sha256[:12],
}])


In [ ]:
# El marco contiene datos protegidos: se usa en memoria y nunca se muestra directamente.
sofa = STORE.read_dataframe(ARTIFACT_NAME)
required = {'sofa_total', 'sofa_complete', 'missing_components'}
missing = required.difference(sofa.columns)
if missing:
    raise ValueError(f'Esquema SOFA incompleto; faltan: {sorted(missing)}')

component_columns = [
    f'sofa_{name}' for name in
    ('respiratory', 'coagulation', 'liver', 'cardiovascular', 'cns', 'renal')
]
missing_components = set(component_columns).difference(sofa.columns)
if missing_components:
    raise ValueError(f'Faltan componentes: {sorted(missing_components)}')


## Completitud agregada

El denominador es el número de horas UCI del artefacto, no el número de pacientes. La completitud estricta exige los seis componentes; `sofa_total` conserva el convenio MIMIC de puntuar como cero los componentes ausentes.

In [ ]:
n_hours = len(sofa)
completeness = pd.DataFrame({
    'component': [column.removeprefix('sofa_') for column in component_columns],
    'hours_observed': [int(sofa[column].notna().sum()) for column in component_columns],
})
completeness['hours_total'] = n_hours
completeness['percent_observed'] = (
    100 * completeness['hours_observed'] / completeness['hours_total']
).round(1)
completeness


In [ ]:
missingness = (
    sofa['missing_components'].value_counts(dropna=False).sort_index()
    .rename_axis('missing_components').rename('hours').reset_index()
)
missingness['percent_hours'] = (100 * missingness['hours'] / n_hours).round(1)
missingness


## Distribución agregada del SOFA

La tabla contiene únicamente frecuencias por puntuación. Se usa como única fuente para ambas implementaciones gráficas.

In [ ]:
distribution = (
    sofa['sofa_total'].value_counts(dropna=False).sort_index()
    .rename_axis('sofa_total').rename('hours').reset_index()
)
distribution['percent_hours'] = (100 * distribution['hours'] / n_hours).round(2)
distribution


### Estilo Python (`matplotlib`)

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise ImportError(
        'Falta matplotlib. Actualiza el entorno con environment.yml o instala .[analysis].'
    ) from exc

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.bar(distribution['sofa_total'].astype(str), distribution['percent_hours'], color='#2878B5')
ax.set(title='Distribución horaria del SOFA (convención MIMIC)', xlabel='SOFA total', ylabel='Horas UCI (%)')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()


### Estilo R (`ggplot2`)

Un notebook tiene un solo kernel principal. Para mantener Python como orquestador y usar el `ggplot2` real (no una emulación), la celda llama al `Rscript` del mismo entorno y muestra el SVG resultante.

In [ ]:
rscript = shutil.which('Rscript')
if rscript is None:
    raise RuntimeError('Rscript no está disponible; recrea el entorno desde environment.yml.')

r_code = r'''
args <- commandArgs(trailingOnly = TRUE)
suppressPackageStartupMessages(library(ggplot2))
d <- read.csv(args[[1]], check.names = FALSE)
d$sofa_total <- factor(d$sofa_total, levels = d$sofa_total)
p <- ggplot(d, aes(x = sofa_total, y = percent_hours)) +
  geom_col(fill = '#2878B5', width = 0.8) +
  labs(title = 'Distribución horaria del SOFA (convención MIMIC)',
       x = 'SOFA total', y = 'Horas UCI (%)') +
  theme_minimal(base_size = 12) +
  theme(panel.grid.minor = element_blank(), plot.title.position = 'plot')
ggsave(args[[2]], p, width = 9, height = 4.8, units = 'in', device = grDevices::svg)
'''
with tempfile.TemporaryDirectory() as directory:
    directory = Path(directory)
    aggregate_csv = directory / 'sofa_distribution_aggregate.csv'
    figure_svg = directory / 'sofa_distribution_ggplot2.svg'
    distribution.to_csv(aggregate_csv, index=False)
    completed = subprocess.run(
        [rscript, '-e', r_code, str(aggregate_csv), str(figure_svg)],
        text=True, capture_output=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(f'ggplot2 falló:\n{completed.stderr}')
    display(SVG(filename=str(figure_svg)))


## Criterio para avanzar

1. El manifiesto y checksum deben validar sin excepciones.
2. La completitud por componente y la distribución deben ser clínicamente plausibles y quedar revisadas.
3. Las diferencias entre gráficos son de estilo; números y denominadores deben coincidir porque comparten la misma tabla agregada.
4. El siguiente paso enlazará sospecha de infección y cambios temporales de SOFA, conservando trazabilidad y análisis de sensibilidad por componentes ausentes.